# Fine-Tuning Spanish RoBERTa for Sentiment Intensity Regression

**Course:** Artificial Intelligence · Unit 6 — Natural Language Processing  
**Institution:** Universidad Tecnológica de Bolívar  
**Reference:** Russell, S. & Norvig, P. (2021). *Artificial Intelligence: A Modern Approach* (4th ed.), Ch. 24 — Deep Learning for NLP  
**Python compatibility:** 3.10 · 3.11 · 3.12 · 3.13

---

## Learning Objectives

1. Fine-tune `PlanTL-GOB-ES/roberta-base-bne` (Spanish RoBERTa) for regression
2. Build a custom PyTorch `Dataset` for Hugging Face `Trainer`
3. Track training loss and evaluate on a held-out test set
4. Visualise predicted vs. real sentiment intensity with scatter plots


---

## 1. Environment Setup

Install once in your terminal:

```bash
pip install transformers[torch] accelerate==0.20.1 pandas scikit-learn matplotlib openpyxl
```


In [ ]:
# pip install transformers[torch] accelerate==0.20.1 pandas scikit-learn matplotlib openpyxl
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from transformers import (
    RobertaForSequenceClassification,
    RobertaTokenizer,
    Trainer,
    TrainingArguments,
)

DIR_DATA = Path.cwd() / 'data'


---

## 2. Load Training Data

The dataset contains Spanish tweets from the TASS / Valence task with  
a continuous `Intensity` label (sentiment strength).


In [ ]:
train_data = pd.read_excel(DIR_DATA / 'Valence_train_oc_es.xlsx')
train_data['Intensity'] = train_data['Intensity'].astype(np.float32)
train_data['Tweet'] = train_data['Tweet'].astype(str)
print(f'Training samples: {len(train_data)}')
train_data.info()


> **Output interpretation:** Check that the 'Tweet' column contains string text and 'Intensity' is a float32 column. The intensity label represents sentiment strength on a continuous scale. Missing values in either column must be handled before tokenisation.


In [ ]:
train_data.head()


> **Output interpretation:** Each row shows a tweet alongside its real-valued intensity score. Higher absolute values indicate stronger sentiment; the sign (if applicable) indicates polarity.


---

## 3. Load the Pre-trained Model and Tokeniser

`PlanTL-GOB-ES/roberta-base-bne` is a Spanish RoBERTa model trained on the  
Spanish National Library corpus (`bne`). `num_labels=1` configures the model  
as a regression head (single continuous output).


In [ ]:
MODEL_NAME = 'PlanTL-GOB-ES/roberta-base-bne'

tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
model = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)
print(f'Model loaded: {MODEL_NAME}')
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')


> **Output interpretation:** Downloading the model weights (~500 MB) happens only on the first run; subsequent runs use the local HuggingFace cache. Running on CPU is possible but slow; a GPU (CUDA or MPS) reduces training time from hours to minutes.


---

## 4. Tokenise and Build PyTorch Dataset


In [ ]:
train_encodings = tokenizer(
    train_data['Tweet'].tolist(),
    truncation=True,
    padding=True,
    return_tensors='pt',
)


class TweetDataset(torch.utils.data.Dataset):
    """PyTorch Dataset wrapping tokenised tweets and float intensity labels."""

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = TweetDataset(train_encodings, train_data['Intensity'].tolist())
print(f'Train dataset size: {len(train_dataset)}')


---

## 5. Configure and Run Training


In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    logging_dir='./logs',
    logging_steps=10,
    do_train=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)


In [ ]:
trainer.train()


> **Output interpretation:** Training loss should decrease across the 3 epochs. For regression (num_labels=1), the Trainer uses MSE loss by default. If loss stagnates after epoch 1, consider reducing the learning rate or increasing epochs.


---

## 6. Training Loss Curve


In [ ]:
train_results = pd.DataFrame(trainer.state.log_history)
loss_rows = train_results.dropna(subset=['loss'])

plt.figure(figsize=(10, 5))
plt.plot(loss_rows['step'], loss_rows['loss'], color='steelblue', label='Training Loss')
plt.xlabel('Training Steps')
plt.ylabel('Loss (MSE)')
plt.title('Training Loss over Steps — Spanish RoBERTa')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


> **Output interpretation:** A smooth decreasing curve indicates stable convergence. Spikes or oscillations may signal that the batch size is too small or the learning rate is too high. The curve should flatten by the final epoch, showing the model has converged.


---

## 7. Load Test Data and Evaluate


In [ ]:
test_data = pd.read_excel(DIR_DATA / 'Valence_test_oc_es.xlsx')
test_data['Intensity'] = test_data['Intensity'].astype(np.float32)
test_data['Tweet'] = test_data['Tweet'].astype(str)

test_encodings = tokenizer(
    test_data['Tweet'].tolist(),
    truncation=True,
    padding=True,
    return_tensors='pt',
)
test_dataset = TweetDataset(test_encodings, test_data['Intensity'].tolist())
print(f'Test dataset size: {len(test_dataset)}')


In [ ]:
eval_trainer = Trainer(model=model, args=training_args)
eval_predictions = eval_trainer.predict(test_dataset)
predicted_intensity = eval_predictions.predictions.squeeze()

real_intensities = np.array(test_data['Intensity'].tolist())
predicted_rounded = np.round(predicted_intensity)

correct = np.sum(real_intensities == predicted_rounded)
accuracy_pct = (correct / len(real_intensities)) * 100
print(f'Exact-match accuracy (rounded): {accuracy_pct:.2f}%')


> **Output interpretation:** Exact-match accuracy on a continuous regression task is a strict metric — even a prediction of 0.49 vs. a label of 0.5 counts as wrong after rounding. Consider also reporting MAE (mean absolute error) and Pearson correlation, which are standard metrics for sentiment intensity regression.


---

## 8. Predicted vs. Real Intensity Scatter Plot


In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(real_intensities, predicted_intensity, alpha=0.5, color='steelblue')
plt.plot(
    [real_intensities.min(), real_intensities.max()],
    [real_intensities.min(), real_intensities.max()],
    'r--',
    label='Perfect prediction',
)
plt.xlabel('Real Intensity')
plt.ylabel('Predicted Intensity')
plt.title('Real vs. Predicted Sentiment Intensity — Spanish RoBERTa')
plt.legend()
plt.tight_layout()
plt.show()


> **Output interpretation:** Points along the red dashed diagonal represent perfect predictions. A dense cluster near the diagonal indicates a well-calibrated model. Systematic vertical spread (actual values spread across a narrow predicted range) signals under-fitting — the model is not capturing the full intensity scale.
